<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex11-vision-navigation/Ex11_vision_navigation_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_11 — Vision-Based Navigation, from A to B

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L11.1 · Vision-Based Navigation and L11.2 · Obstacle Avoidance**

A JetRacer has to drive from A to B along a taped line, past obstacles
standing on it, as fast as it safely can, deciding from its camera alone. The
two lectures ask four questions on the way, and this notebook answers each one
with a measurement:

| | the question | lecture |
|---|---|---|
| **where is the path?** | the position of the line a little way ahead, from one frame | L11.1 |
| **how do I steer onto it?** | the steering that follows, and how often it must be decided | L11.1 |
| **is the path blocked?** | free or blocked, from one frame | L11.2 |
| **what do I do about it?** | stop, steer around, or follow a potential field | L11.2 |

**Everything runs here, on a virtual JetRacer**: a digital twin of the car
with the same software interface (`car.steering`, `car.throttle`,
`camera.read()`), whose 224 × 224 colour frames are rendered from where the
car stands on the floor. The true answer to every question is therefore known.
The same task on the real JetRacer is a demonstration in the lab; the mini
projects run on the lab's Yahboom ROSMASTER R2, with its built-in depth
camera and LiDAR, and an ultrasonic sensor as an optional add-on.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the JetRacer and its twin, the frames on the course, the classical line detector | — |
| **4** | a convolutional network that finds the line, trained on two collections of frames | TODO 1 — the network |
| **5** | the same network saying free or blocked, and its threshold | TODO 2 — the cross-entropy |
| **6** | three avoidance strategies on the course from A to B | TODO 3 — the potential field |
| **7** | the network driving the twin, the stopping distance, the GPU and fewer bits | — |
| **8** | the race: a conventional and a deep-learning JetRacer from A to B | — |
| **9** | what the notebook says | — |
| **10 – 11** | your report, and the mini project proposals | your answers |

About six minutes on a CPU, most of it training. **Use a GPU runtime on Colab**
(Runtime → Change runtime type → T4 GPU): the training is several times faster,
and section 7 compares the two. Numbers are printed with at most two decimals,
except errors smaller than that.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex11-vision-navigation/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import copy
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex11_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
torch.set_default_dtype(torch.float32)                   # frames need no double precision
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The task from A to B

### The JetRacer Pro

The car of the lab, as Waveshare specifies it:

| | |
|---|---|
| computer | NVIDIA Jetson Nano 4 GB: four Cortex-A57 cores, a 128-core Maxwell GPU, 472 GFLOPS |
| camera | IMX219-160: 8 MP sensor, 160° wide-angle lens, 3280 × 2464 pixels |
| steering | Ackermann: the front wheels turn, driven by a 6 kg·cm servo |
| drive | RC380 brushed motor, four-wheel drive through front and rear differentials |
| battery | four 18650 cells, 2S2P, 8.4 V |
| speed sensor | **none**: there is no wheel encoder, so speed is timed over a known distance |
| software | the JetRacer library: `car.steering` and `car.throttle`, each from −1 to 1; `CSICamera(width=224, height=224)` |

On the car, the road-following notebook reads a frame, lets a network give the
position of the line in it, `x`, from −1 (left) to +1 (right), and steers by
`car.steering = x * STEERING_GAIN`. The servo then receives
`car.steering * steering_gain + steering_offset` (the library's defaults are
−0.65 and 0), and the motor a duty of `car.throttle * throttle_gain` (0.8).

### The virtual JetRacer

`pb.VirtualJetRacer` is that car in software, with the same interface. Set
`car.steering` and `car.throttle`, call `car.step(dt)`, and the car moves;
`car.camera.read()` returns the 224 × 224 colour frame its camera sees from
where it stands. Inside it:

* **the camera** is a pinhole 10 cm above the floor, tilted 20° down, with a
  120° field of view - a stand-in for the 160° lens, which bends straight lines;
* **the steering**: the servo command turns the front wheels by up to 35° (at
  most 30°), and the kinematic bicycle model with a wheelbase $L$ = 0.20 m moves
  the car; a positive `car.steering` turns right, as on the car;
* **the speed** approaches 5 m/s × the motor duty with a time constant of 0.3 s,
  and changes no faster than the tyres' friction (0.6) allows.

**Everything in that list is assumed, to be measured on the real car**; the
vendor does not state it.

**The steering law.** The car aims at the line a distance $\ell$ = 0.40 m ahead
of its camera. Pure pursuit (L11.1, *Classical Pipeline*) gives the wheel
angle that reaches an aim point at an angle $\alpha$ to the heading:

$$\delta = \arctan\!\Big(\frac{2L\sin\alpha}{\ell}\Big)$$

and the road-following law `car.steering = x * STEERING_GAIN` is its linear
form, with the aim point's position in the frame, `x`, standing for $\alpha$.

**Between two decisions the car is blind.** At speed $v$ and $f$ decisions a
second it drives

$$d = \frac{v}{f}$$

on its last command (L11.1, *Why a GPU*).

**The course** is 6 m long, along a taped line from A to B. Three obstacles of
radius 8 cm stand on or beside the line, 1.5 m apart; the shaded ring round each
is the car's half-width, 10 cm, so the car's centre must stay outside it. An
obstacle **blocks** the path when it stands in the car's lane within 0.7 m.
The course is assumed too, about the size of the lab's floor.

The cell prints the problem, drives the twin for two seconds with the steering
held, and shows its first frame.

In [ ]:
pb.describe_problem()
car = pb.VirtualJetRacer()                                    # the twin, at A, facing B
car.throttle = pb.throttle_for(1.0)                           # the throttle that settles at 1 m/s
car.steering = 0.3                                            # a little to the right
path = []
for _ in range(1000):                                         # 2 s in steps of 2 ms
    car.step(0.002)
    path.append((car.x, car.y))
print(f"after 2 s: x = {car.x:.2f} m, y = {car.y:.2f} m, heading {np.degrees(car.th):.0f} degrees, speed {car.v:.2f} m/s")

fig = plt.figure(figsize=(13, 7.4))
pb.view3d(0.0, 0.0, 0.0, fig.add_axes([0.02, 0.36, 0.45, 0.62], projection="3d", computed_zorder=False),
          title="the virtual JetRacer at A")                  # the twin in 3-D, seen from behind
a = fig.add_axes([0.55, 0.40, 0.40, 0.55])
a.imshow(pb.VirtualJetRacer().camera.read()); a.axis("off")   # what its camera sees there
a.set_title("camera.read() at A: 224 x 224", fontsize=9)
a = fig.add_axes([0.06, 0.08, 0.90, 0.24])
pb.draw_course(a, paths={"steering 0.3, throttle for 1 m/s": path})
a.set_title("the course from A to B, and the twin driven with the steering held", fontsize=9)
plt.show()

**What you should see.** The numbers of the problem; the twin at A in 3-D,
seen from behind and a little to its left - a drawing after the vendor's
photograph to the twin's own numbers, not a measurement of the car - and its
camera's frame; a path that bends right
on a circle (the wheel angle held, the speed reaching 1 m/s within about a
second), 58° round after 2 s; and the camera's frame at A: the line running ahead, the first
obstacle 1.5 m away, and the wall above the horizon.

---

## 2 · The frames on the course

**Every frame is a picture of the course from one place.** Put the car
somewhere on the floor, pointing somewhere, and the frame is what its camera
sees from there: the floor it covers, the line where it crosses that floor, and
any obstacle standing in it. The top view below puts the car at four places on
the course; the blue dots are the floor its camera sees within 1.5 m. Under it,
the car at each place in 3-D, seen from behind as in the film of section 6,
and under that the frame its camera sees there, with the two answers it holds:

* **where is the path?** The position of the line in the row of the frame that
  shows the floor 0.40 m ahead of the camera (the red line), from −1 (the left
  edge) to +1 (the right edge): the **target**. It is a regression (L11.1,
  *Steering From a Frame*), and it is the label the car's road-following
  notebook takes by a click.
* **is the path blocked?** 1 when an obstacle stands in the car's lane within
  0.7 m, 0 otherwise: a classification (L11.2, *Free or Blocked*).

The frames are 224 × 224 × 3, red-green-blue, 0 – 255: exactly what the car's
camera hands its network.

In [ ]:
POSES = [(0.40, 0.00, 0.00), (1.00, 0.03, 0.10), (2.00, -0.12, 0.25), (3.95, 0.12, -0.20)]   # (x, y, heading) on the course
fig = plt.figure(figsize=(13, 9.6))
pb.draw_course(fig.add_axes([0.05, 0.71, 0.90, 0.27]), poses=POSES)
for i, (x, y, th) in enumerate(POSES):
    pb.view3d(x, y, th, fig.add_axes([0.01 + 0.25 * i, 0.35, 0.23, 0.33], projection="3d", computed_zorder=False),
              title=str(i + 1))                               # the car at place i + 1, in 3-D
    frame = pb.VirtualJetRacer(x, y, th, obstacles=pb.OBSTACLES, seed=i).camera.read()   # what the camera sees there
    b, _, _ = pb.is_blocked(x, y, th, pb.OBSTACLES)
    a = fig.add_axes([0.03 + 0.25 * i, 0.01, 0.19, 0.30])
    a.imshow(frame); a.axhline(pb.ROW, color="r", lw=0.6)
    a.plot((pb.W - 1) / 2 * (1 + pb.target(x, y, th)), pb.ROW, "r+", ms=12, mew=2)   # the target, in pixels
    a.set_title(f"{i + 1}: target {pb.target(x, y, th):+.2f}, " + ("blocked" if b else "free"), fontsize=9); a.axis("off")
plt.show()

**What you should see.** From place 1, on the line, the line runs straight up
the frame and the first obstacle stands 1.1 m ahead, beyond 0.7 m: free. From
place 2, turned a little left, the line leans to the right and the obstacle is
half a metre ahead: blocked. From place 3, to the right of the line and turned
towards it, the line comes in from the left and crosses the red row near the
middle, and the next obstacle is a metre away: free. From place 4, to the left
of the line and turned towards it, the third obstacle stands just over half a
metre ahead: blocked. The further away a thing stands, the higher and smaller
it is in the frame, and the horizon is the row where the floor ends.

**Three sets of test frames** score the steering, all on the lap of section 7
(a stadium, so the frames see turns as well as straights), 300 each, never
trained on: with the car on the line (`line`), beside it and turned away
(`off-line`), and as off-line with glare - a window reflected in the floor -
on every frame (`glare`). They are taken anywhere the car may be. The training
frames come from a person driving, in section 4.

---

## 3 · The classical detector

**The classical way to find the line** needs no training (L11.1, *Classical
Pipeline*). Make the frame grey, take a few rows round the red row, estimate
the floor's brightness by the median, keep what is brighter than the floor by
0.15, take the largest bright run of columns, and return its
brightness-weighted mean column:

$$\hat{x} = \frac{\sum_j w_j\,x_j}{\sum_j w_j}, \qquad w_j = \max(I_j - I_{\mathrm{floor}} - 0.15,\ 0)$$

`pb.detect(pb.shrink(frames, k))` does that, on the frame reduced by $k$ in
each direction.

**Its cost is set by the resolution.** `pb.shrink` replaces every $k \times k$
block of pixels by its mean. A smaller frame is cheaper to read out and to
process, and blurs the tape.

**The agreed accuracy is 0.03**: a mean error of about three pixels of the
224-pixel frame. The cell tests the detector at four resolutions on 300 frames
of each collection.

In [ ]:
TEST = {kind: pb.steering_frames(300, kind, seed=100 + i) for i, kind in enumerate(("line", "off-line", "glare"))}   # held out: never trained on
AGREED = 0.03

def mean_error(predict, kind):
    x, y = TEST[kind]
    return float(np.mean(np.abs(predict(x) - y)))             # the mean distance from the true target

classic = {}
for k in (1, 2, 4, 8):
    classic[k] = [mean_error(lambda x: pb.detect(pb.shrink(x, k)), kind) for kind in TEST]
    ok = max(classic[k][:2]) <= AGREED
    print(f"  {pb.H // k:3d} x {pb.W // k:3d} pixels: line {classic[k][0]:.3f}, off-line {classic[k][1]:.3f}, glare {classic[k][2]:.3f}"
          + ("   <- meets 0.03 without glare" if ok else ""))
K = 2                                                         # the coarsest frame that meets it
frame1 = TEST["line"][0][:1]                                  # one frame, for the timing
t_classic = middle(lambda: pb.detect(pb.shrink(frame1, K)))
print(f"  the detector on one 112 x 112 frame: {1000 * t_classic:.2f} ms")

**The failure, seen whole.** One held-out glare frame, drawn with what the
rule does to it: the frame with the detector's band of rows, the true column
of the tape in green, and, as the orange cross, the aim the weighted mean
picks - the formula of L11.1's *Where Conventional Vision Fails*, the mean
column of every pixel brighter than the floor, each counted by its
brightness. Under it, column for column, the band row's brightness: the tape
makes the narrow peak, the glare the wide bump, and the mean lands between
them. The notebook's own detector is one refinement smarter - it keeps only
the **largest** bright run - which saves this frame, but the table above
shows it still fails on the collection: with glare it is wrong by 0.08 to
0.15 at every resolution.

In [ ]:
info = pb.show_glare_failure()                                # the 80th-percentile glare frame: frame, band, tape and the aim the weighted mean picks

**What you should see.** Without glare the detector is within 0.03 at 224 and
at 112 pixels across (0.005 to 0.026), and fails below: at 28 × 28 the tape is
blurred into the floor. **With glare it is wrong by 0.08 to 0.15 at every
resolution**:
the glare is bright, and bright is all it looks for (L11.1, *Where Conventional
Vision Fails*).

---

## 4 · The steering network, trained on a human drive

**A convolution** is the operation of L5.1: a small kernel of weights $w$ slid
over the frame, each output the sum of products of the kernel with the pixels
under it, plus a bias,

$$y_{c}(i, j) = b_c + \sum_{c'} \sum_{m=-2}^{2}\sum_{n=-2}^{2} w_{c,c'}(m, n)\; x_{c'}(i + m,\ j + n)$$

with $c'$ the three colours in and $c$ the channel out. The same kernel is used
at every place of the frame, so a kernel that answers to a bright stripe
answers to the tape wherever it lies: that is why a convolutional network, and
not a dense one, reads a frame (L11.1, *Steering From a Frame*). A stride of 2
moves the kernel two pixels at a time and halves the frame.

**The network** takes the 224 × 224 frame and returns the target. Four
convolutional layers, each halving the frame, then two dense layers:

| layer | output |
|---|---|
| convolution 5 × 5, stride 2, 8 channels, ReLU | 8 × 112 × 112 |
| convolution 3 × 3, stride 2, 16 channels, ReLU | 16 × 56 × 56 |
| convolution 3 × 3, stride 2, 32 channels, ReLU | 32 × 28 × 28 |
| convolution 3 × 3, stride 2, 32 channels, ReLU | 32 × 14 × 14 |
| flatten, dense 32, leaky ReLU | 32 |
| dense, `outputs` numbers | the answer |

**Why the dense layer's ReLU is leaky.** A ReLU outputs zero for every
negative input, and its gradient there is zero too (L4.1). If one large early
step pushes all 32 neurons of the dense layer negative on every frame, none of
them can learn again: the network returns the same number for every frame,
and the car drives in a circle. With a plain ReLU this happened on several of
the random starts we tried, within the first five steps. A **leaky ReLU**,
$\max(0.01\,z,\ z)$, keeps a small slope below zero, so every neuron keeps a
gradient; with it every start we tried trained (UDL ch. 3 names the leaky
ReLU among the activations).

**The same network answers both questions of the block**: with one output it
finds the line (here), with two it says free or blocked (section 5). The car's
own network is a ResNet-18 trained on millions of photographs, with a new
last layer (L11.2, *Transfer Learning*); this one is small enough to train from
nothing here.

**The loss** is the mean squared distance between the network's target and the
true one, averaged over the training frames (L11.1, *Steering From a Frame*).

**Two drives, 1500 frames each.** A person drives the car round the lap with
a gamepad, and the camera saves a frame every 0.1 s, each labelled with the
true target. That is how the JetRacer's own road-following notebook collects
its data, and how the survey cars of a road map drive every road once to
record it. `pb.human_drive` simulates the person (a pure-pursuit driver with
a hand's wobble). One drive is **smooth**: the car stays on the line. The
other **swerves on purpose**, up to about 20 cm from the line and back, with
the sun reflected on the floor in a third of its frames (L11.1, *Learning
From a Human Driver*).

**TODO 1** is the network.

In [ ]:
class Net(torch.nn.Module):                                   # a network: layers in __init__, the computation in forward
    def __init__(self, outputs):
        super().__init__()
        nn = torch.nn
        self.body = nn.Sequential(                            # the layers, applied one after the other
            nn.Conv2d(3, 8, 5, stride=2, padding=2), nn.ReLU(),     # convolution: 3 colours in, 8 channels out, 112 x 112
            nn.Conv2d(8, 16, 3, stride=2, padding=1), nn.ReLU(),    # convolution: 16 channels, 56 x 56
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),   # convolution: 32 channels, 28 x 28
            nn.Conv2d(32, 32, 3, stride=2, padding=1), nn.ReLU(),   # convolution: 32 channels, 14 x 14
            nn.Flatten(),                                     # the 32 x 14 x 14 block as one row of 6272 numbers
            nn.Linear(32 * 14 * 14, 32), nn.LeakyReLU(0.01),  # dense layer: 6272 inputs -> 32 outputs; leaky, so it cannot die
            nn.Linear(32, outputs))                           # dense layer: 32 inputs -> the answer
    def forward(self, x):
        return self.body(x.float() / 255)                     # pixels 0 - 255 -> 0 - 1, then the layers

In [ ]:
def train(outputs, x, y, loss_fn, epochs=20, batch=64):
    # a network with `outputs` outputs, trained on the frames x and answers y
    set_seed(88)                                              # fix the random start so a run repeats
    net = Net(outputs).to(DEVICE)                             # the network, on the device
    X, Y = torch.as_tensor(x).to(DEVICE), torch.as_tensor(y).to(DEVICE)   # frames stay 0 - 255 bytes until used
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)         # Adam: adaptive steps for every weight (at 2e-3 the steering network can collapse to one answer)
    t0 = time.perf_counter()
    for _ in range(epochs):                                   # one epoch: every frame seen once
        order = torch.randperm(len(X), device=DEVICE)         # the frames in a new random order
        for i in range(0, len(X), batch):
            b = order[i:i + batch]                            # one mini-batch of 64 frames
            opt.zero_grad()                                   # clear the gradients of the step before
            loss = loss_fn(net(X[b]), Y[b])                   # how far the answers are from the truth
            loss.backward()                                   # backpropagation: the gradient of the loss
            opt.step()                                        # one Adam step on the weights
    return net, time.perf_counter() - t0

def outputs(model, x):
    # the network's outputs for NumPy frames, 256 at a time
    with torch.no_grad():  # no gradient tracking: evaluation only
        return np.concatenate([to_numpy(model(torch.as_tensor(x[i:i + 256]).to(DEVICE))) for i in range(0, len(x), 256)])   # the network on 256 frames at a time

def squared(out, y):
    return (out[:, 0] - y).pow(2).mean()                      # mean squared distance from the true target

def steer_net(style):
    x, y = pb.human_drive(1500, style, seed=1)                # a person drives the lap: 150 s, a frame every 0.1 s
    return train(1, x, y, squared)

net_smooth, sec_smooth = steer_net("smooth")                  # the drive that stays on the line
net, sec = steer_net("swerving")                              # the drive that swerves away and back on purpose
print(f"{parameter_count(net)} weights; trained in {sec_smooth:.0f} s and {sec:.0f} s")

def run(model):
    # the steering network as a function of NumPy frames, for mean_error
    return lambda x: outputs(model, x)[:, 0]

In [ ]:
err_smooth = [mean_error(run(net_smooth), kind) for kind in TEST]
err_net = [mean_error(run(net), kind) for kind in TEST]
x1 = torch.as_tensor(frame1).to(DEVICE)                       # one frame, for the timing
with torch.no_grad():  # no gradient tracking: evaluation only
    t_net = middle(lambda: net(x1))

print(f"{'mean error on 300 test frames':40s}{'line':>8s}{'off-line':>10s}{'glare':>8s}{'one frame':>12s}{'training':>10s}")
print(f"  {'classical detector, 112 x 112':38s}{classic[K][0]:8.3f}{classic[K][1]:10.3f}{classic[K][2]:8.3f}{1000 * t_classic:9.2f} ms{'-':>10s}")
print(f"  {'network, smooth drive':38s}{err_smooth[0]:8.3f}{err_smooth[1]:10.3f}{err_smooth[2]:8.3f}{1000 * t_net:9.2f} ms{sec_smooth:8.0f} s")
print(f"  {'network, drive with swerves':38s}{err_net[0]:8.3f}{err_net[1]:10.3f}{err_net[2]:8.3f}{1000 * t_net:9.2f} ms{sec:8.0f} s")

M_PER_UNIT = (np.nanmax(pb.FLOOR_L[pb.ROW]) - np.nanmin(pb.FLOOR_L[pb.ROW])) / 2   # metres across the floor 40 cm ahead per unit of the target

def rms_cm(predict, kind):
    x, y = TEST[kind]
    return 100 * M_PER_UNIT * float(np.sqrt(np.mean((predict(x) - y) ** 2)))   # root mean square, in cm on the floor

print(f"\n{'root mean square error, cm on the floor':40s}{'line':>8s}{'off-line':>10s}{'glare':>8s}")
print(f"  {'classical detector, 112 x 112':38s}" + "".join(f"{rms_cm(lambda x: pb.detect(pb.shrink(x, K)), k):{w}.1f}" for k, w in zip(TEST, (8, 10, 8))))
print(f"  {'network, drive with swerves':38s}" + "".join(f"{rms_cm(run(net), k):{w}.1f}" for k, w in zip(TEST, (8, 10, 8))))

x, y = TEST["glare"]
fig, ax = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
for a, (name, pred) in zip(ax, (("classical detector", pb.detect(pb.shrink(x, K))),
                                ("network, smooth drive", run(net_smooth)(x)), ("network, drive with swerves", run(net)(x)))):
    a.plot([-1, 1], [-1, 1], "k--", lw=1); a.plot(y, pred, ".", ms=3)
    a.set_title(name + ", frames with glare", fontsize=9); a.set_xlabel("true target")
ax[0].set_ylabel("predicted target"); plt.tight_layout()
plt.savefig(cc.output_path("Ex11_report_1.png"), dpi=150); plt.show()

**What you should see.** Three rows that fail in three different places.

* **The classical detector** is within 0.03 without glare and wrong by about
  0.08 with it.
* **The network trained on the smooth drive** is within about 0.01 with the
  car on the line, and wrong by about 0.2 the moment the car is beside the
  line - which is exactly when the answer matters. Its driver never left the
  line, so it has never seen the line anywhere but in front of it.
* **The network trained on the drive with swerves** is within about 0.02 on the
  line, 0.04 beside it and 0.05 with glare. Same network, same number of
  frames, same training time. Only the drive differs.

**As a distance on the floor.** The second table gives the same comparison
as a root mean square, the squared error the network was trained on, in
centimetres across the floor 40 cm ahead (one unit of the target is about
70 cm there). The truth is exact: the twin draws each frame from a pose it
knows and computes where the tape crosses the row. The detector is closer on
the line (about 0.8 cm against 1.8); the network is closer beside the line
(3.6 against 4.5) and with glare (5.3 against 14.1), because the root mean
square weighs the detector's large misses. These are the numbers on L11.1,
*Steering From a Frame*.

**What the first layer learned.** The cell below draws the eight 5 × 5
kernels of the swerving drive's network, first convolution, summed over the three
colours, and what each makes of one frame: eight feature maps of 112 × 112,
before the ReLU. Red is a positive weight or answer, blue a negative one.

**Where the rule goes wrong, frame by frame.** The cell below takes, from
the frames beside the line and the frames with glare, the two on which the
classical detector is furthest from the truth, and marks on each the true
crossing of the tape with the row 40 cm ahead (green circle), the
detector's aim (orange cross) and the network's (magenta plus). The title
gives both errors as distances across the floor at that row (L11.1, *Where
Conventional Vision Fails*).

In [ ]:
cases = []
for kind in ("off-line", "glare"):
    x, y = TEST[kind]
    d_aim = pb.detect(pb.shrink(x, K))                        # the detector's aim on every test frame
    n_aim = run(net)(x)                                       # the network's
    for i in np.argsort(-np.abs(d_aim - y))[:2]:              # the two frames the detector misses most
        cases.append((kind, x[i], y[i], d_aim[i], n_aim[i]))
column = lambda t: np.clip((pb.W - 1) / 2 * (1 + t), 0, pb.W - 1)   # a target, -1 .. 1, as a pixel column
fig, ax = plt.subplots(1, 4, figsize=(13, 3.7))
for a, (kind, f, t, d, n) in zip(ax, cases):
    a.imshow(f.transpose(1, 2, 0)); a.axhline(pb.ROW, color="w", lw=0.6)   # the frame and the row 40 cm ahead
    a.plot(column(t), pb.ROW, "o", mfc="none", mec="lime", ms=13, mew=2)    # the truth
    a.plot(column(d), pb.ROW, "x", color="orange", ms=12, mew=3)          # the detector
    a.plot(column(n), pb.ROW, "+", color="magenta", ms=15, mew=2.5)       # the network
    a.set_title(f"{kind}: detector {100 * M_PER_UNIT * abs(d - t):.0f} cm off,\nnetwork {100 * M_PER_UNIT * abs(n - t):.0f} cm off", fontsize=9)
    a.axis("off")
plt.tight_layout(); plt.show()

**What you should see.** On the frames beside the line the detector locks on
a far piece of tape or on nothing at all and aims tens of centimetres off;
with glare it locks on the reflection, which is brighter than the floor and
as wide as the tape. The network, trained on frames that held both, stays
within a few centimetres on most of them: it has learned what the tape looks
like, not only that it is bright.

In [ ]:
w = to_numpy(net.body[0].weight).sum(axis=1)                  # the first layer's kernels, summed over the colours: 8 x 5 x 5
frame = TEST["off-line"][0][:1]                               # one frame, beside the line
with torch.no_grad():  # no gradient tracking: evaluation only
    maps = to_numpy(net.body[0](torch.as_tensor(frame).to(DEVICE).float() / 255))[0]   # 8 feature maps, before the ReLU
fig, ax = plt.subplots(2, 9, figsize=(14, 3.4))
ax[0, 0].imshow(frame[0].transpose(1, 2, 0)); ax[0, 0].set_title("the frame", fontsize=8)
ax[1, 0].axis("off")
for c in range(8):
    a = np.abs(w[c]).max()
    ax[0, c + 1].imshow(w[c], cmap="RdBu_r", vmin=-a, vmax=a)   # the kernel: red positive, blue negative
    ax[0, c + 1].set_title(f"kernel {c + 1}", fontsize=8)
    m = maps[c] - np.median(maps[c])                          # the answer, measured from the floor's
    b = np.abs(m).max()
    ax[1, c + 1].imshow(m, cmap="RdBu_r", vmin=-b, vmax=b)
for a in ax.flat:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

**What you should see.** In every map the tape stands out from the floor, in
red where the kernel's weights add up to a positive number and in blue where
they add up to a negative one, and the wall stands apart from the floor too.
Most of these kernels answer to brightness more than to an edge: on these
frames the tape is the brightest thing on the floor, and that is what training
found useful. Nobody wrote these kernels; a Sobel kernel, set by hand (−1, 0, +1 across
the columns, the middle row counted twice: the edge kernel of L5.1, which lit
up along the crack in the weld radiographs), would answer only at the tape's
two edges. That answers *where is the path*.

---

## 5 · The obstacle network

**Is the path blocked?** The classical answer is a rule: on the floor of the
car's lane, up to a metre ahead, is anything darker than the floor by 0.15?
`pb.dark_rule(frames)` does that. A shadow is dark too, which is where it
fails one way; an obstacle as light as the floor is not dark, which is where
it fails the other (L11.2, *Identifying an Obstacle*).

**The frames.** `pb.obstacle_frames` puts the twin at random places near the
line on the A-to-B course with, on four frames in five, an obstacle somewhere
ahead - a dark box (half of them), a grey box as light as the floor, or a low
dark block - and on two frames in five a shadow. The label comes from where the
obstacle stands: blocked when it is in the lane within 0.7 m.

**The network** is section 4's with two outputs, $z_{\mathrm{f}}$ for free and
$z_{\mathrm{b}}$ for blocked. The softmax turns them into the probability that
the path is blocked (L11.2, *Free or Blocked*):

$$p_{\mathrm{b}} = \frac{e^{z_{\mathrm{b}}}}{e^{z_{\mathrm{f}}} + e^{z_{\mathrm{b}}}}$$

and the loss is the **cross-entropy**, the mean over a batch of $N$ frames of
minus the log-probability the network gives the true label $y_k$:

$$\mathcal{L} = -\frac{1}{N}\sum_{k=1}^{N}\left[y_k \ln p_{\mathrm{b},k} + (1 - y_k)\ln(1 - p_{\mathrm{b},k})\right]$$

It is trained on 2000 frames and tested on 800 others.

**TODO 2** is the cross-entropy. Compute it from the log of the softmax,
`torch.log_softmax`, which is exact where the softmax itself would round a
tiny probability to zero.

In [ ]:
def cross_entropy(out, y):
    # out: the two outputs per frame (N, 2); y: the true label, 0 free or 1 blocked (N,)
    logp = torch.log_softmax(out, dim=1)                      # ln p_f and ln p_b of every frame, by the softmax
    return -logp[torch.arange(len(y)), y].mean()              # minus the mean log-probability of the true label

In [ ]:
xo, bo, _, _ = pb.obstacle_frames(2000, seed=1)               # training frames: obstacles, shadows, and their labels
obs_net, sec_obs = train(2, xo, bo, cross_entropy, epochs=15)
xt, bt, wt, kt = pb.obstacle_frames(800, seed=2)              # held out: never trained on
p_blocked = to_numpy(torch.softmax(torch.as_tensor(outputs(obs_net, xt)), 1))[:, 1]   # the softmax's probability of blocked
shadow_free = bt == 0                                         # the free frames, some with a shadow

def rates(pred):
    # right, missed obstacles (blocked called free), false stops (free called blocked)
    return (float(np.mean(pred == bt)), float(np.mean(pred[bt == 1] == 0)), float(np.mean(pred[bt == 0] == 1)))

def missed(pred, kind):
    return float(np.mean(pred[(bt == 1) & (kt == kind)] == 0))   # missed obstacles of one kind

rule = pb.dark_rule(xt)
cnn = (p_blocked > 0.5).astype(int)
print(f"{'800 held-out frames':26s}{'right':>8s}{'missed':>9s}{'false stops':>13s}{'missed: dark':>14s}{'grey':>7s}{'low':>7s}")
for name, pred in (("dark-pixel rule", rule), ("network, threshold 0.5", cnn)):
    r = rates(pred)
    print(f"  {name:24s}{100 * r[0]:7.1f}%{100 * r[1]:8.1f}%{100 * r[2]:12.1f}%"
          + "".join(f"{100 * missed(pred, k):{w}.1f}%" for k, w in (("box", 13), ("grey", 6), ("low", 6))))
print(f"blocked on {100 * bt.mean():.0f} % of the frames; network trained in {sec_obs:.0f} s")

**What you should see.** The rule is right on about 85 % of the frames: it
stops for about one free frame in seven - the shadows - and misses about a fifth
of the obstacles, seven grey boxes in ten. The network is right on about 97 %
and stops for almost no shadow. **It misses grey boxes too**, about four in
ten: one camera barely sees an obstacle that is as light as the floor,
whatever reads the frame. That is what a second sensor is for, in the mini
projects.

**The threshold.** The car treats the path as blocked when $p_{\mathrm{b}}$ is
above a threshold $p_{\mathrm{th}}$. One half is not the right value, because the two
errors do not cost the same: a missed obstacle is a collision, a false stop a
second lost. With $c_{\mathrm{miss}}$ and $c_{\mathrm{stop}}$ the costs, the
expected cost of a threshold is (L11.2, *The Cost of an Error*)

$$C(p_{\mathrm{th}}) = c_{\mathrm{miss}}\,P_{\mathrm{miss}}(p_{\mathrm{th}}) + c_{\mathrm{stop}}\,P_{\mathrm{stop}}(p_{\mathrm{th}})$$

The cell sweeps $p_{\mathrm{th}}$ and takes the cheapest, with a collision costed **100
times** a false stop.

In [ ]:
C_MISS, C_STOP = 100.0, 1.0                                   # a collision costs 100 false stops
thresholds = np.round(np.linspace(0.05, 0.95, 19), 2)
miss = np.array([rates((p_blocked > t).astype(int))[1] for t in thresholds])
stop = np.array([rates((p_blocked > t).astype(int))[2] for t in thresholds])
cost = C_MISS * miss + C_STOP * stop
P_TH = float(thresholds[np.argmin(cost)])
P_MISS, P_FALSE = rates((p_blocked > P_TH).astype(int))[1:]
print(f"cheapest threshold {P_TH:.2f}: missed {100 * P_MISS:.1f} %, false stops {100 * P_FALSE:.1f} %, "
      f"grey boxes missed {100 * missed((p_blocked > P_TH).astype(int), 'grey'):.1f} %")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(thresholds, 100 * miss, "o-", ms=3, label="missed obstacles")
ax.plot(thresholds, 100 * stop, "s-", ms=3, label="false stops")
ax.axvline(P_TH, color="k", ls="--", lw=1)
ax.set_xlabel(r"threshold $p_{\mathrm{th}}$ on $p_\mathrm{b}$"); ax.set_ylabel("% of frames"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**What you should see.** Lowering $p_{\mathrm{th}}$ trades missed obstacles for false
stops, and because a miss costs a hundred times more, the cheapest threshold is
the lowest tried, 0.05: the missed obstacles fall from about 15 to 4 %, the
grey boxes from four in ten to about one in ten, and the false stops rise from
almost none to about 5 %.
That answers *is the path blocked*.

---

## 6 · The avoidance strategies

**What do I do about it?** Three strategies, each needing more from the
perception than the one before (L11.2, *Deciding the Action*):

| strategy | when blocked | what it must be told |
|---|---|---|
| **stop** | throttle to zero until the path is free | free or blocked |
| **steer around** | aim at a lane 0.30 m to the side away from the obstacle, slowed to 60 %, until 0.5 m past it | free or blocked, and which side |
| **potential field** | aim, one aim distance ahead, at the lowest point of $U$ | where each obstacle stands |

**The potential field** adds a push away from every obstacle within range to a
pull back to the line (L11.2, *Steering Around*):

$$U(y) = y^2 + \sum_i \frac{K}{d_i(y)}, \qquad y^{*} = \arg\min_y U(y)$$

$y$ is the sideways position one aim distance $\ell$ ahead, $d_i(y)$ the gap
there between the car's edge and obstacle $i$ (at least 1 cm), and $K$ =
0.03 m the strength of the push. The car aims at $y^{*}$.

`pb.avoid(strategy, f)` drives the twin from A to B through `car.steering` and
`car.throttle`, deciding `f` times a second; `strategy` is `"stop"`,
`"steer"`, or a function `aim(x, near)` that returns $y^{*}$ for a car at `x`
and the obstacles `near` it (rows of $(x, y, r)$).

**TODO 3** is that function.

In [ ]:
def field_aim(x, near, K=pb.K_FIELD):
    # the sideways position to aim at, one aim distance ahead of a car at x
    ahead = x + pb.AIM                                        # where the aim point lies along the course
    ys = np.linspace(-0.6, 0.6, 241)                          # candidate sideways positions, 5 mm apart
    U = ys ** 2                                               # the pull back to the line
    for (ox, oy, r) in near:                                  # one push for every obstacle within range
        d = np.maximum(np.hypot(ahead - ox, ys - oy) - r - pb.CAR_HALF, 0.01)   # the gap from the car's edge
        U = U + K / d
    return float(ys[np.argmin(U)])                            # where U is lowest

In [ ]:
STRATEGIES = {"stop": "stop", "steer around": "steer", "potential field": field_aim}
runs = {}
print(f"{'strategy':18s}{'decisions':>11s}{'time to B':>12s}{'smallest gap':>15s}")
for name, s in STRATEGIES.items():
    for f in (5, 10, 30):
        r = pb.avoid(s, f)                                    # perfect perception: the true free or blocked
        runs[name, f] = r
        when = f"{r['time']:.2f} s" if r["reached"] else "never"
        print(f"  {name:16s}{f:8d} Hz{when:>12s}{100 * r['clearance']:12.1f} cm" + ("   <- collision" if r["clearance"] < 0 else ""))

# the same at 10 Hz, perceived by section 5's network: its own missed and false rates, 20 runs each
print(f"with the network's errors at threshold {P_TH:.2f} (missed {100 * P_MISS:.1f} %, false {100 * P_FALSE:.1f} %), 20 runs at 10 Hz:")
for name, s in list(STRATEGIES.items())[1:]:
    rr = [pb.avoid(s, 10, p_miss=P_MISS, p_false=P_FALSE, seed=k) for k in range(20)]
    t = [r["time"] for r in rr if r["reached"]]
    print(f"  {name:16s} collisions {sum(r['clearance'] < 0 for r in rr):2d} of 20, mean time to B {np.mean(t):.2f} s")

fig, ax = plt.subplots(figsize=(11, 3))
pb.draw_course(ax, paths={name: runs[name, 10]["path"] for name in STRATEGIES})
ax.set_title("the three strategies at 10 decisions a second", fontsize=9)
plt.tight_layout(); plt.savefig(cc.output_path("Ex11_report_2.png"), dpi=150); plt.show()

**The film of the run.** The cell below films the potential field's run at
10 Hz: the JetRacer in 3-D, seen from behind and above as it drives from A to
B without stopping, beside the frame its camera sees at each moment and the
course from above. It takes about a minute to draw; set `FILM = False` to skip
it.

In [ ]:
FILM = True
if FILM:
    anim = pb.film(runs["potential field", 10], "Virtual JetRacer: the potential field, A to B")   # one picture every 1/25 s
    try:
        from IPython.display import HTML, display
        display(HTML(anim.to_html5_video()))                    # in the notebook; needs ffmpeg, which Colab has
    except ImportError:                                         # outside a notebook: the film as a file
        anim.save(cc.output_path("Ex11_film.mp4"), fps=25)

**What you should see.**

* **Stop never reaches B**: the first obstacle stands on the line and does not
  move, so the car stands in front of it until the 30 s run ends.
* **Steer around reaches B in about 8.3 s**, with half a centimetre to spare
  at 10 Hz: it holds a fixed side lane, slows, and turns only when the obstacle
  is already within 0.7 m.
* **The potential field reaches B in about 6.3 s** with about 7 cm to spare:
  it knows where each obstacle stands, never slows, and bends the path only as
  much as the gap needs.

With the network's error rates in place of perfect perception, steering around
collides on two runs in twenty - a missed frame is all its half-centimetre can
take - and the field on none.
That answers *what do I do about it*: the strategy decides what the perception
must deliver, and the richer answer - where, not only whether - is what buys
the time.

---

## 7 · The loop rate, the GPU and fewer bits

**The network drives the twin.** `pb.drive_line(policy, f, v)` puts the
virtual car on the lap - a stadium of 7.20 m, straights of 1.4 m and turns of
0.7 m radius, the size of the vendor's track map - at speed $v$, and $f$ times
a second hands `car.camera.read()` to `policy`, the network trained on the drive with swerves, and
sets `car.steering = x * STEERING_GAIN` from its answer, as the car's
road-following notebook does. Computing a decision takes one period, so the
wheels follow the command from the frame before. The cell reports the largest
distance from the line; half the narrowest gap of a typical course is 17.5 cm.

**The stopping distance.** From the frame that shows an obstacle to standing
still, the car drives one blind distance, then brakes as hard as the tyres
allow (L11.2, *Stopping Distance*). Both must fit inside the 0.7 m the camera
judges:

$$d_{\mathrm{stop}} = \frac{v}{f} + \frac{v^2}{2\mu g} \leq R$$

with a friction coefficient $\mu$ = 0.6, assumed. The cell finds the largest
safe speed at three loop rates.

**The GPU** (L11.1, *Why a GPU*). Every output of a convolution is its own sum
of products over a small window, independent of the others, so a GPU computes
them at the same time. The cell counts the multiply-adds of one frame and times
one frame and a batch of 256 on the CPU and, if this runtime has one, on the
GPU.

**Fewer bits** (L11.2, *Deployment on the Car*). On a car the network's
weights are stored in fewer bits so that a decision takes less time and
memory. `pb.round_weights(net, b)` rounds every weight onto $2^b$ levels; the
cell measures what that does to both networks.

In [ ]:
def policy(frame):
    return float(outputs(net, pb.to_net(frame[None]))[0, 0])   # one camera frame -> the network's target

print(f"{'speed':>8s}{'decisions':>12s}{'blind':>9s}{'furthest from the line':>26s}")
loop = []
for v, f in ((1.0, 30), (2.0, 30), (2.0, 20), (1.0, 10), (2.0, 15), (2.0, 12), (2.0, 10), (1.0, 5)):
    dev, _ = pb.drive_line(policy, f, v)                      # two laps of the twin, the network steering
    loop.append((v / f, dev))
    print(f"{v:6.1f} m/s{f:9d} Hz{100 * v / f:7.1f} cm{100 * dev:21.2f} cm" + ("   <- off the course" if dev > pb.HALF_CLEARANCE else ""))

speeds = np.linspace(0, 5, 5001)
safe = {f: float(speeds[pb.stopping_distance(speeds, f) <= pb.RANGE].max()) for f in (5, 10, 30)}
print("largest safe speed within the camera's 0.7 m: " + ", ".join(f"{s:.2f} m/s at {f} Hz" for f, s in safe.items()))

In [ ]:
def multiply_adds(model):
    # count the multiply-adds of one frame, layer by layer
    count = []
    def hook(layer, inp, out):
        if isinstance(layer, torch.nn.Conv2d):
            count.append(out[0].numel() * layer.in_channels * layer.kernel_size[0] * layer.kernel_size[1])   # every output: in-channels x kernel area
        elif isinstance(layer, torch.nn.Linear):             # every output: one product per input
            count.append(layer.in_features * layer.out_features)
    hooks = [m.register_forward_hook(hook) for m in model.modules()]   # listen to every layer
    with torch.no_grad():  # no gradient tracking: evaluation only
        model(torch.zeros(1, 3, pb.H, pb.W, device=next(model.parameters()).device))
    for h in hooks:
        h.remove()
    return sum(count)

print(f"one frame through the steering network: {multiply_adds(net):,} multiply-adds")
frames_cpu = torch.as_tensor(TEST["off-line"][0][:256])       # 256 frames on the CPU
devices = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
gpu_time = {}
for dev in devices:
    m = copy.deepcopy(net).to(dev).eval()                     # the same network on this device
    one, batch = frames_cpu[:1].to(dev), frames_cpu.to(dev)
    sync = torch.cuda.synchronize if dev == "cuda" else (lambda: None)   # wait for the GPU before reading the clock
    with torch.no_grad():  # no gradient tracking: evaluation only
        t1 = middle(lambda: (m(one), sync()))
        t256 = middle(lambda: (m(batch), sync()))
    gpu_time[dev] = (t1, t256)
    print(f"  {dev:4s}: one frame {1000 * t1:.2f} ms, 256 frames {1000 * t256:.2f} ms, {1000 * t256 / 256:.3f} ms a frame")
if len(devices) == 1:
    print("  no GPU on this runtime: Runtime -> Change runtime type -> T4 GPU, and run the notebook again")

bits = (8, 6, 5, 4, 3, 2)
err_bits = [mean_error(run(pb.round_weights(net, b)), "off-line") for b in bits]   # the steering network, its weights rounded
acc_bits = []
for b in bits:
    p = to_numpy(torch.softmax(torch.as_tensor(outputs(pb.round_weights(obs_net, b), xt)), 1))[:, 1]   # p_b, weights in b bits
    acc_bits.append(rates((p > P_TH).astype(int))[0])
print("weights in " + ", ".join(str(b) for b in bits) + " bits:")
print("  steering, mean error off the line: " + ", ".join(f"{e:.3f}" for e in err_bits) + f"  (as trained: {err_net[1]:.3f})")
print("  obstacles, right: " + ", ".join(f"{100 * a:.1f} %" for a in acc_bits) + f"  (as trained: {100 * rates((p_blocked > P_TH).astype(int))[0]:.1f} %)")

**The whole decision, three ways.** One decision is one frame in and one
steering command out. The cell below times it for the classical detector on
the CPU (112 × 112, as in section 3), for the network on the CPU and, if the
runtime has one, for the network on the GPU, one frame at a time as the car
drives. From each time it gives the most decisions a second the computer
alone allows, the blind distance at 1 m/s and at 2 m/s (the decision is used
one period later, section 7), and beside them each method's error with the
car beside the line and with glare (root mean square, cm across the floor
40 cm ahead). On the car the camera, the transfer and the servo add their
own time, which is measured there (L11.1, *Why a GPU*).

In [ ]:
rows = [("detector, CPU", t_classic, lambda x: pb.detect(pb.shrink(x, K))),
        ("network, CPU", gpu_time["cpu"][0], run(net))]
if "cuda" in gpu_time:
    rows.append(("network, GPU", gpu_time["cuda"][0], run(net)))
print(f"{'one decision':16s}{'time':>10s}{'at most':>12s}{'blind at 1 m/s':>16s}{'at 2 m/s':>10s}{'beside line':>13s}{'glare':>8s}")
for name, t, f in rows:
    print(f"  {name:14s}{1000 * t:8.2f} ms{1 / t:9.0f} Hz{100 * 1.0 * t:13.2f} cm{100 * 2.0 * t:7.2f} cm"
          f"{rms_cm(f, 'off-line'):10.1f} cm{rms_cm(f, 'glare'):6.1f} cm")
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh([r[0] for r in rows], [1000 * r[1] for r in rows], color=["tab:orange", "tab:blue", "tab:green"][:len(rows)])
ax.set_xscale("log"); ax.set_xlabel("time for one decision [ms], log scale"); ax.invert_yaxis()
plt.tight_layout(); plt.show()
if "cuda" not in gpu_time:
    print("no GPU on this runtime: Runtime -> Change runtime type -> T4 GPU, and run the notebook again")

**What you should see.** On a CPU the detector is the fastest of the three,
under a millisecond, and the network on the same CPU takes about twice as
long; on a laptop both allow hundreds of decisions a second. The detector
buys its speed with its error: beside the line and with glare it is several
centimetres further off than the network. On Colab's GPU one frame of the
network takes about as long as on the CPU, because one small frame does not
keep the GPU's cores busy; what the GPU buys is the batch (the 256 frames of
section 7 in a fraction of the CPU's time) and a network ten or a hundred
times larger, as the car's own ResNet-18, at the same rate. On the Nano,
whose CPU is much slower than a laptop's, the GPU is what makes the network
fast enough. Write down your own numbers: they are the runtime's, not the
car's.

**What you should see.**

* **The blind distance decides, not the speed or the loop rate alone.** With
  the network steering, the twin stays within about 5 cm of the line up to
  17 cm blind and swings 15 cm at 20 cm - close to the 17.5 cm half-gap;
  1 m/s at 10 Hz and 2 m/s at 20 Hz are both 10 cm blind and give about the
  same deviation.
* **The safe speed rises with the loop rate**: about 1.9 m/s at 5 Hz, 2.3 at
  10 Hz and 2.7 at 30 Hz within the 0.7 m the camera judges. The braking term
  does not depend on the network at all; the blind term is all the loop rate
  can buy.
* **One frame is about 17 million multiply-adds** - a ResNet-18, the car's own
  network, needs 1.8 billion. On a CPU one frame takes about a millisecond, and
  a batch is no cheaper a frame: the cores are already busy. On a GPU one frame
  alone is hardly faster, because starting the work costs more than the work,
  but a batch of 256 costs a small fraction of that a frame. On a car the
  frames arrive one at a time, which is why the network is compiled for the
  device and its rate measured there.
* **Fewer bits.** The steering network keeps its accuracy down to 5 bits and
  loses it at 4, because every rounding error lands in the answer. The
  obstacle network keeps it down to 4 bits and loses it at 3: a class only
  changes when the rounding moves $p_{\mathrm{b}}$ across the threshold.

---

## 8 · The race

**Two virtual JetRacers race from A to B**, each on its own copy of a course
that holds everything sections 3 to 6 found difficult: a dark box, a grey box
the colour of the floor, another dark box, two shadows that are not
obstacles, and two patches of sun on the floor (`pb.RACE_COURSE`, assumed).
Each car decides only from its own camera, ten times a second:

| | finds the line with | sees an obstacle with |
|---|---|---|
| **conventional** | the classical detector of section 3 (112 × 112) | the dark-pixel rule of section 5 |
| **deep learning** | the steering network of section 4 | the obstacle network of section 5, at its threshold $p_{\mathrm{th}}$ |

**Both use the same decision rule**: follow the line; when the path is seen
blocked, pass the obstacle ahead in a lane 35 cm beside it at 60 % speed, then steer back
towards the line and follow it again. Where
the obstacle stands is taken from the course for both cars alike, so the race
compares what the two cars **see**, nothing else. A car that touches an
obstacle stops there; one that strays 60 cm from the line has left the
course. `pb.race` runs it; `pb.race_film` films it, and the
winner is celebrated at B. The film takes about a minute to draw.

In [ ]:
drivers = [
    dict(name="conventional",
         follow=lambda f: pb.detect(pb.shrink(f, K)),                                   # the line: the classical detector
         blocked=lambda f: bool(pb.dark_rule(f)[0])),                                   # an obstacle: the dark-pixel rule
    dict(name="deep learning",
         follow=lambda f: outputs(net, f)[:, 0],                                        # the line: the steering network
         blocked=lambda f: bool(torch.softmax(torch.as_tensor(outputs(obs_net, f)), 1)[0, 1] > P_TH)),   # an obstacle: p_b above the threshold
]
results = pb.race(drivers)                                    # the same course, the same rule, each car with its own eyes
for r in results:                                             # each car's outcome, and when it saw a block
    print(f"  {r['name']:14s} {r['result']}; saw a block {len(r['events'])} times: " + ", ".join(f"{t:.1f} s ({e})" for t, e in r["events"]))

RACE_FILM = True
if RACE_FILM:
    anim = pb.race_film(results)                              # the race, then the award
    try:
        from IPython.display import HTML, display
        display(HTML(anim.to_html5_video()))
    except ImportError:
        anim.save(cc.output_path("Ex11_race.mp4"), fps=25)

**What you should see.** The conventional car follows the line and passes
the first dark box, but the grey box is as light as the floor: the dark-pixel
rule does not see it in time, and the car runs into it. The deep-learning car
sees all three, passes them and reaches B, and is celebrated. Look at the
moments each car saw a block: a block seen where none stands sends a car
round nothing and costs it time. The course was chosen to hold what the rules
were shown to miss, so the result is an illustration, not a statistic -
sections 3 to 6 measured the same differences on hundreds of frames.

---

## 9 · What the notebook says

* **The race puts it in one picture.** With the same decision rule, the car
  that sees with rules ran into the grey box; the car that sees with
  networks reached B. The difference was only in what each saw.
* **The data decided the result, not the network.** Two identical steering
  networks, the same number of frames and the same training time: one fails
  beside the line and in glare, the other does not. Frames from where the car
  should not be are the ones that teach it to come back.
* **A rule fails on what its designer did not think of.** The line detector
  took glare for the line; the dark-pixel rule took shadows for obstacles.
  Both need no data, are the fastest, and are exact on the frames they were
  designed for.
* **One camera has a limit no network removes.** An obstacle as light as the
  floor was missed by the rule and by the network. A second sensor that
  measures distance - the mini projects' - sees it.
* **The threshold is a decision about costs, not a property of the network.**
  With a collision costing a hundred false stops, it went well below one half.
* **The strategy decides what the perception must deliver.** Stopping needs
  only free or blocked and never reached B; steering around reached it with
  half a centimetre to spare and collided when the network missed a frame; the
  potential field needs where each obstacle stands and was fastest, with the
  widest gap.
* **The loop rate is a speed limit.** The blind distance limits how well the
  car follows the line and how fast it may drive towards an obstacle.
* **The twin is not the car.** The camera's mount, the steering and speed
  response, the friction and the course are assumed, and the timings are this
  runtime's. The JetRacer in the lab, and the mini projects on the
  ROSMASTER R2, are where they are measured.

---

## 10 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex11_report.md` with the results of sections 3 to 7 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L11.1 and L11.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "The classical line detector and the network were scored on the same held-out frames. Where did each meet the agreed error, where did each fail, and why does the network not take the glare for the line? (-> L11.1 Q1)": """
""",
    "The two steering networks differ only in the frames they were trained on. What did the first one never see, why did its own held-out error not show the problem, and what would you collect on the real car because of it? (-> L11.1 Q2)": """
""",
    "With the network driving the twin, 1 m/s at 10 Hz and 2 m/s at 20 Hz gave the same result. What is the quantity that decides, at what value did the car leave the course, and what are the two ways to reduce it? (-> L11.1 Q3)": """
""",
    "How many multiply-adds does one 224 x 224 frame take, how did one frame and a batch of 256 compare on the CPU and on the GPU, and what would you measure on the car before trusting a loop rate? (-> L11.1 Q4)": """
""",
    "What did the dark-pixel rule call blocked that was not, and what did it miss? Which obstacles did the network miss as well, and why can one camera not see them? (-> L11.2 Q1)": """
""",
    "Which threshold did the expected cost choose, and how would it move if a collision cost ten false stops instead of a hundred? (-> L11.2 Q2)": """
""",
    "Why did stopping never reach B, and what does the potential field need to be told that steering around does not? (-> L11.2 Q3)": """
""",
    "What is the largest safe speed at 5, 10 and 30 decisions a second, and which of the two terms of the stopping distance does a faster network shorten? (-> L11.2 Q4)": """
""",
    "To conclude: which avoidance strategy reaches B fastest without a collision, and what limits its speed: the errors of the networks, the loop rate, or the distance needed to react? (-> L11.1 and L11.2, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| finding the line: mean error on 300 held-out frames | line | off-line | glare | one frame |", "|---|---|---|---|---|",
            f"| classical detector, 112 x 112 | {classic[K][0]:.3f} | {classic[K][1]:.3f} | {classic[K][2]:.3f} | {1000 * t_classic:.2f} ms |",
            f"| network, smooth drive | {err_smooth[0]:.3f} | {err_smooth[1]:.3f} | {err_smooth[2]:.3f} | {1000 * t_net:.2f} ms |",
            f"| network, drive with swerves | {err_net[0]:.3f} | {err_net[1]:.3f} | {err_net[2]:.3f} | {1000 * t_net:.2f} ms |",
            "", "| free or blocked: 800 held-out frames | right | missed | false stops | grey boxes missed |", "|---|---|---|---|---|",
            *[f"| {name} | {100 * r[0]:.1f} % | {100 * r[1]:.1f} % | {100 * r[2]:.1f} % | {100 * missed(pred, 'grey'):.1f} % |" for name, pred, r in
              (("dark-pixel rule", rule, rates(rule)), ("network, threshold 0.5", cnn, rates(cnn)),
               (f"network, threshold {P_TH:.2f}", (p_blocked > P_TH).astype(int), rates((p_blocked > P_TH).astype(int))))],
            "", "| strategy, 10 Hz | time to B | smallest gap |", "|---|---|---|",
            *[f"| {name} | {runs[name, 10]['time']:.2f} s | {100 * runs[name, 10]['clearance']:.1f} cm |" if runs[name, 10]["reached"]
              else f"| {name} | never | {100 * runs[name, 10]['clearance']:.1f} cm |" for name in STRATEGIES],
            "", "Blind distance against the furthest the twin got from the line, the network steering: "
            + ", ".join(f"{100 * d:.1f} cm -> {100 * e:.2f} cm" for d, e in sorted(loop)),
            "Largest safe speed: " + ", ".join(f"{s:.2f} m/s at {f} Hz" for f, s in safe.items()) + " (friction coefficient 0.6 assumed)",
            f"One frame: {multiply_adds(net):,} multiply-adds; " + "; ".join(f"{d}: one frame {1000 * a:.2f} ms, 256 frames {1000 * b:.2f} ms" for d, (a, b) in gpu_time.items()),
            "Steering network, weights in " + ", ".join(str(b) for b in bits) + " bits: mean error " + ", ".join(f"{e:.3f}" for e in err_bits),
            "", "The frames are rendered by the virtual JetRacer; its camera, steering, speed response and friction, and the course, are assumed."]
    out = ["# Ex_11 — Vision-Based Navigation, from A to B", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex11_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures of sections 4 and 6, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex11_report.md into Ex11_report.pdf, with any figure
# saved as Ex11_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex11_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex11_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex11_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex11_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 11 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP11A · Faster from A to B, on the ROSMASTER R2

**The problem.** In the notebook's race both cars pass every obstacle by the same slow rule:
slow to 60 %, swerve into a lane beside it, come back. It is safe, and it is
what made the winning car take 8.5 s for 6 m. Here you make the lab's
**Yahboom ROSMASTER R2** faster from A to B with its camera alone, by better obstacle
avoidance: a network that says not only *blocked* but *where* the obstacle
stands, and a way round it that does not brake - the potential field of
section 6, or a path and a speed you plan yourself - at the highest speed
the stopping distance allows.

| | |
|---|---|
| car | the lab's Yahboom ROSMASTER R2, Superior version: Jetson Orin Nano Super 8 GB, Ackermann steering, 65 mm wheels, at most 1.8 m/s, ROS2 Humble (vendor); its Astra Pro Plus depth camera, of which this project uses the colour image only, resized to 224 × 224 |
| course | in the lab: a taped line of 6 m from A to B; four obstacles at least 0.9 m apart, within 15 cm of the line - dark boxes 20 cm tall, a grey box the colour of the floor; two dark sheets of paper flat on the floor, which are not obstacles |
| ground truth | the obstacles' positions, measured with a tape to 1 cm; they label every frame you record with *blocked* and *where* |
| speed | your choice, justified by the stopping distance of L11.2 ($v/f + v^2/(2\mu g)$ within what the camera judges), and at most the R2's 1.8 m/s |

Before the car, the work is done on the notebook's virtual JetRacer
(`pb.make_course`, `pb.run_course` with `decide=pb.FieldAround()`), whose
geometry gives the true position of every obstacle. The twin's camera mount,
steering and speed response are a JetRacer's and assumed (`problem.py`):
measure the R2's camera height and tilt and set them there before you train on
rendered frames.

**Why a plain PINN is not enough.** Speed comes from two places: not braking round an obstacle, and driving
faster between them. Both need to know where the obstacle is, early and
reliably - a car that knows only *blocked* has to swerve wide and slow. And
the faster the car, the further ahead it must see, where the frame is
coarsest (L11.1, *The JetRacer Pro*).

**What you build.** The notebook's network with a second head: blocked or free (cross-entropy) and
the obstacle's forward and sideways position (squared error, only on blocked
frames). Its positions feed the potential field (`pb.FieldAround`) or your own
planner; raise the speed while the twin's closed loop stays without collision.
Then train or fine-tune on frames recorded on the R2, and put it in the
car's loop through the R2's ROS2 drive interface (Yahboom's course
materials), in place of the twin's `car.steering` and `car.throttle`.

**The ground truth you get.** On the floor, the course you lay out and measure. On the twin,
`MP11A_truth.npz` from `tools/miniprojects/ex11_truth.py A`: perfect
perception driven on 20 held-out courses with the steer-around rule and with
the potential field at 1, 1.5 and 2 m/s - the times you are held to - and the
camera's classical rule on 1500 held-out poses; labels are exact geometry, and
the drive agrees on time steps of 4, 2 and 1 ms.

**Worked example.** 

| on the twin, 20 held-out courses | collisions | mean time to B |
|---|---|---|
| perfect perception, steer around, 1 m/s | 0 | 8.76 s |
| perfect perception, potential field, 1 m/s | 0 | 6.34 s |
| perfect perception, potential field, 1.5 m/s | 0 | 4.24 s |
| perfect perception, potential field, 2 m/s | 5 | 3.20 s |
| camera's dark-pixel rule, steer around, 1 m/s | 20 | 8.79 s |
| camera's dark-pixel rule, potential field, 1.5 m/s | 20 | 4.16 s |

The field with perfect perception is twice as fast as the race's rule; at
2 m/s even perfect perception collides. The classical camera rule collides
on every course: speed without knowing where is no speed at all.

**Requirements.**

* On the twin: **no collision** on the 20 held-out courses with your network's perception, and a mean time to B at most **5.0 s** (perfect perception with the field at 1.5 m/s: 4.24 s).
* On the twin: the position head within **5 cm** (root mean square) of the true obstacle position on the held-out blocked poses.
* On the car: **five runs** from A to B on your measured course with **no contact**, filmed; the time of each, against the notebook rule's time on the same course.

**What you hand in.** The twin's times and gaps against the table above, the position head's error,
the speed you chose and why the stopping distance allows it, the five runs on
the car with their film, and what differed between the twin and the floor.

### MP11B · Depth camera and LiDAR in difficult conditions, on the ROSMASTER R2

**The problem.** The notebook's car sees with one camera, and the race showed what it misses:
an obstacle as light as the floor. A dim room, sun on the floor and low
obstacles make it worse. Here you use the **ROSMASTER R2**'s built-in
sensors, its depth camera and its LiDAR, with a DFRobot URM09 ultrasonic
sensor as an optional add-on, and make one network read them together, so
that the car drives from A to B in difficult conditions: grey boxes, low
blocks, shadows, glare and a dim room. Each sensor fails differently - the
LiDAR does not see below its scan plane, the colour image does not see grey
in a dim room, the depth image sees shape but not colour, and the ultrasonic
sensor says how far but not where.

| | |
|---|---|
| car | as MP11A |
| LiDAR | the R2's SLAM A1 (Slamtec RPLIDAR A1): 360 ranges, one a degree, 0.15 to 12 m, 5.5 scans a second, on USB (vendor); scan plane height on the R2 not stated (measure yours) |
| ultrasonic, optional | an add-on DFRobot URM09 on the Orin Nano's I2C bus (the R2 carries none of its own): 2 to 500 cm, 1 cm steps, accuracy 1 % (vendor); cone ±30°; about 20 readings a second, mounted 4 cm high on the bumper (assumed: measure yours) |
| depth camera | the R2's built-in Astra Pro Plus: a colour and a depth image (vendor); resolution and range on this car not stated (measure yours) |
| course | as MP11A, with more grey boxes and a low block 5 cm high, and the conditions made hard: the room's lights dimmed, a lamp or the sun on the floor beside the line, dark paper as shadows |
| ground truth | the obstacles' positions, measured with a tape to 1 cm; they label every frame and reading you record |

Before the car, the fusion is written on the notebook's virtual JetRacer,
which simulates the scan and the ultrasonic range from the same geometry
(`pb.lidar`, `pb.sonar`, `pb.run_course`) and the hard conditions with
`pb.make_course(seed, hard=True)`. The twin renders no depth image, so the
depth branch is trained on recordings from the R2.

**Why a plain PINN is not enough.** No sensor sees every obstacle, and joining their rules by OR inherits every
false stop of each and still does not say where an obstacle only the
ultrasonic sensor heard stands. A network that reads all three can learn when
to believe which.

**What you build.** One network with a branch per sensor - convolutions over the colour and
the depth image, a 1-D convolution over the scan (which wraps round at 360°),
and the ultrasonic range joined before the dense layers if you add it - and two heads, blocked or free and where the
obstacle stands. First on the twin's hard courses, then trained or fine-tuned
on recordings from the car in the dimmed lab, and put in the car's loop.

**The ground truth you get.** On the floor, the course you lay out and measure. On the twin,
`MP11B_truth.npz` from `tools/miniprojects/ex11_truth.py B`: the classical
rules of each sensor and of all three on 1500 held-out poses of hard courses,
and perfect perception driven on 20 held-out hard courses, the reference time;
labels and ranges are exact geometry, and the drive agrees on time steps of
4, 2 and 1 ms.

**Worked example.** 

| classical rule, twin, 1500 held-out poses of hard courses | missed: dark box | grey box | low block | stops for a shadow | right |
|---|---|---|---|---|---|
| camera, dark pixels in the lane | 28.7 % | 96.8 % | 20.5 % | 2.4 % | 76.7 % |
| LiDAR, a return in the lane | 25.0 % | 12.1 % | 100 % | 14.2 % | 70.5 % |
| ultrasonic, a range under 0.7 m | 22.5 % | 23.0 % | 23.2 % | 5.5 % | 86.6 % |
| camera OR LiDAR OR ultrasonic | 11.2 % | 3.2 % | 14.4 % | 19.7 % | 84.5 % |

Driven on the 20 held-out hard courses, perfect perception reaches B in
8.70 s on average with at least 3.8 cm to spare; the camera rule and the
ultrasonic rule collide on all 20, the LiDAR rule on 14, all three by OR on
12.

**Requirements.**

* On the twin's hard courses: at most **5 %** of each kind of obstacle missed and at most **5 %** of the shadows called blocked on the held-out poses; **no collision** on the 20 held-out hard courses, with a mean time to B within **10 %** of 8.70 s.
* On the twin: the network with each branch removed in turn, to show what each sensor contributes.
* On the car, in the dimmed lab with the lamp on the floor: **five runs** from A to B on your measured course with **no contact**, filmed; the time of each and the smallest gap you can measure.

**What you hand in.** The twin's numbers against the classical rules, what each sensor contributed,
the five runs on the car with their film, and what differed between the twin
and the floor.